<!-- torchleet:colab -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fengyee/TorchLeet/blob/main/v4/multimodal/vlm-connector/vlm-connector.ipynb)

Check your work: `pip install -e python/` once from the repo root (on Colab: `!pip install -q "git+https://github.com/Fengyee/TorchLeet#subdirectory=python"`), then `from torchleet import check; check("vlm-connector", ...)`


# VLM Connector: MLP Projector, Perceiver Resampler and Image-Token Merging

**Difficulty**: 🟡 Medium

### Problem Statement

A vision-language model (LLaVA, Flamingo, Idefics, PaliGemma, Qwen-VL, …) is three parts: a **vision encoder** (a CLIP/SigLIP ViT that turns an image into a grid of patch features, e.g. $24 \times 24 = 576$ tokens of width 1024 for ViT-L/14 at 336 px), a **connector** that maps those features into the LLM's embedding space, and the **LLM**. The prompt contains `<image>` placeholder tokens, and before the first transformer layer their embeddings are replaced by the connector's outputs.

There are two families of connector:

- **MLP projector** (LLaVA-1.5's `mlp2x_gelu`): a per-token `Linear → GELU → Linear`. All 576 visual tokens reach the LLM, so no detail is lost, but every image costs 576 positions of context.
- **Perceiver Resampler** (Flamingo, IDEFICS): a fixed set of **learned latent queries** cross-attends to however many image tokens there are and always outputs `num_queries` tokens (64 in Flamingo). BLIP-2's Q-Former is a heavier relative of this.

You will build both, plus the **merge** step that splices image features into the text sequence and builds the language-modelling labels.

**Perceiver Resampler layer** (pre-LayerNorm, residual). With latents $L \in \mathbb R^{Q \times d}$ and image tokens $X \in \mathbb R^{N \times d}$:

$$\tilde X = \mathrm{LN}(X),\quad \tilde L = \mathrm{LN}(L),\quad q = \tilde L W_q,\quad [k, v] = [\tilde X;\ \tilde L]\,W_{kv}$$

$$L \leftarrow L + \mathrm{MHA}(q, k, v)\,W_o,\qquad L \leftarrow L + \mathrm{MLP}(\mathrm{LN}(L))$$

$[\tilde X; \tilde L]$ concatenates along the **token** axis: Flamingo lets the latents attend to themselves as well as to the image. Using only $\tilde X$ for keys/values is the common simpler variant, and the grader accepts both. The reference solution implements Flamingo's concatenation. A final LayerNorm is applied after the last layer. This simplified version adds **no positional encoding to the image tokens**, so the output is invariant to the order of the image tokens.

---

### Requirements

1. **`MLPProjector(vision_dim, llm_dim)`** (`nn.Module`): exactly two `nn.Linear` layers, `Linear(vision_dim, llm_dim) → GELU → Linear(llm_dim, llm_dim)`, applied to every token independently. `forward(x)`: `(B, T, vision_dim) → (B, T, llm_dim)`.
2. **`PerceiverResampler(dim, num_queries, num_heads, num_layers=1)`** (`nn.Module`):
   - `self.latents`: an `nn.Parameter` of shape `(num_queries, dim)` with a random init (e.g. `torch.randn`).
   - `num_layers` blocks, each a pre-LN multi-head cross-attention (`num_heads` heads; `dim % num_heads == 0`) with a residual connection, followed by a pre-LN MLP (`Linear(dim, 4·dim) → GELU → Linear(4·dim, dim)`) with a residual connection. LayerNorm the image tokens **and** the latents before the attention projections. Then a final `LayerNorm`.
   - `forward(x)`: `(B, N, dim) → (B, num_queries, dim)` for **any** `N`. Samples in a batch must not interact.
3. **`merge_image_features(input_ids, text_embeds, image_features, image_token_id, attention_mask=None, ignore_index=-100) -> (inputs_embeds, labels)`**
   - `input_ids`: `(B, L)` int64. Each row contains exactly `T` copies of `image_token_id` (one image per sample), at a different offset in each row (prompts differ, padding differs).
   - `text_embeds`: `(B, L, D)`, the LLM embedding lookup of `input_ids` (the rows at placeholders hold the meaningless embedding of `<image>`).
   - `image_features`: `(B, T, D)`, already projected. `image_features[b, k]` goes to the **k-th placeholder of sample b**.
   - `inputs_embeds`: `(B, L, D)` in **`text_embeds`' dtype** (cast the features: vision tower in fp32, LLM in bf16). Text positions are unchanged. Return a new tensor and don't modify any input in place. Gradients must flow to `image_features` and to `text_embeds`.
   - `labels`: `(B, L)` int64, equal to `input_ids` with placeholder positions **and** padding positions (`attention_mask == 0`, when a mask is given) set to `ignore_index`. **No shift**: the LM loss shifts later.
   - Raise **`ValueError`** if any row's number of placeholders differs from `T`.

---

### Constraints

- ✅ `F.scaled_dot_product_attention` or `nn.MultiheadAttention(batch_first=True)` are fine for the attention.
- ❌ No positional embeddings on the image tokens, and no pooling or slicing by position in the resampler.
- ❌ No Python loop over tokens **or samples** in `merge_image_features`: use a boolean mask.
- ❌ Use `raise ValueError`, not `assert`, for the count check (asserts vanish under `python -O`).

---

<details>
  <summary>💡 Hint</summary>

  - Projector: what shape does `nn.Linear` accept? Does it care about the extra `T` dimension?
  - Resampler: the latents are shared by every image. How do you give each sample in the batch its own copy without copying memory? Which tensor provides the queries, and which the keys/values?
  - A plain `.sum()` of a LayerNorm output has (almost) zero gradient. When you test gradient flow, use a random weighting.
  - Merge: `input_ids == image_token_id` is a `(B, L)` boolean mask. In what order does `x[mask]` visit the `True` positions, and how does that relate to `image_features.reshape(-1, D)`?
  - Why is the per-row count check needed even when the total count matches?
  - Padding id 0 can also be a real token. Which input tells you reliably where padding is?

</details>

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
# A toy VLM: the "vision encoder" emits 16 patch features of width 64 per image; the "LLM"
# has a 1000-token vocabulary and hidden size 96. Token 999 is the <image> placeholder.
torch.manual_seed(0)
B, N_PATCH, VISION_DIM, LLM_DIM, VOCAB = 2, 16, 64, 96, 1000
IMAGE_TOKEN_ID, PAD_ID = 999, 0
vision_features = torch.randn(B, N_PATCH, VISION_DIM)   # stand-in for a CLIP/SigLIP ViT output
embed_tokens = nn.Embedding(VOCAB, LLM_DIM)             # the LLM's input embedding table


def make_batch(n_image_tokens):
    """Two prompts with the image at different offsets; sample 1 is left-padded."""
    a = [5, 17, 42, 1, 2] + [IMAGE_TOKEN_ID] * n_image_tokens + [7, 8, 9, 10]
    b = [3, 4] + [IMAGE_TOKEN_ID] * n_image_tokens + [13, 14]
    pad = len(a) - len(b)
    input_ids = torch.tensor([a, [PAD_ID] * pad + b])
    attention_mask = torch.tensor([[1] * len(a), [0] * pad + [1] * len(b)])
    return input_ids, attention_mask


ids, mask = make_batch(4)
print(ids)
print(mask)

In [ ]:
class MLPProjector(nn.Module):
    """LLaVA-1.5 vision-to-language projector: Linear -> GELU -> Linear, applied per token."""

    def __init__(self, vision_dim: int, llm_dim: int):
        super().__init__()
        # TODO: Linear(vision_dim, llm_dim), GELU, Linear(llm_dim, llm_dim)
        raise NotImplementedError

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, T, vision_dim) -> (B, T, llm_dim)"""
        # TODO
        raise NotImplementedError

In [ ]:
torch.manual_seed(0)
proj = MLPProjector(VISION_DIM, LLM_DIM)
out = proj(vision_features)
assert out.shape == (B, N_PATCH, LLM_DIM), f"got {tuple(out.shape)}"
linears = [m for m in proj.modules() if isinstance(m, nn.Linear)]
assert [(l.in_features, l.out_features) for l in linears] == [(VISION_DIM, LLM_DIM), (LLM_DIM, LLM_DIM)], \
    "expected Linear(vision_dim, llm_dim) then Linear(llm_dim, llm_dim)"

# Per token: shuffling the tokens shuffles the outputs the same way.
perm = torch.randperm(N_PATCH)
assert torch.allclose(proj(vision_features[:, perm]), out[:, perm], atol=1e-6)

# Not affine: for an affine map f(x1 + x2) - f(x2) == f(x1) - f(0).
x1, x2, zero = torch.randn(1, 1, VISION_DIM), torch.randn(1, 1, VISION_DIM), torch.zeros(1, 1, VISION_DIM)
assert not torch.allclose(proj(x1 + x2) - proj(x2), proj(x1) - proj(zero), atol=1e-4), \
    "the projector is affine -- put a GELU between the two Linear layers"
print("MLPProjector tests passed")

In [ ]:
class PerceiverResampler(nn.Module):
    """
    Flamingo-style Perceiver Resampler: num_queries learned latent vectors cross-attend to a
    variable number of image tokens and come out as exactly num_queries visual tokens.
    """

    def __init__(self, dim: int, num_queries: int, num_heads: int, num_layers: int = 1):
        super().__init__()
        # TODO: self.latents = nn.Parameter of shape (num_queries, dim), random init
        # TODO: num_layers blocks, each with: LayerNorms for the image tokens and the latents,
        #       q / kv / out projections (or nn.MultiheadAttention(batch_first=True)),
        #       and a pre-LN MLP (dim -> 4*dim -> dim)
        # TODO: a final LayerNorm
        raise NotImplementedError

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, N, dim) image tokens, any N -> (B, num_queries, dim)"""
        # TODO: give every sample the same latents (expand), run the blocks, final norm
        raise NotImplementedError

In [ ]:
torch.manual_seed(0)
NUM_QUERIES = 8
resampler = PerceiverResampler(LLM_DIM, NUM_QUERIES, num_heads=4, num_layers=2)
for n in (1, 16, 100):
    out = resampler(torch.randn(B, n, LLM_DIM))
    assert out.shape == (B, NUM_QUERIES, LLM_DIM), f"{n} image tokens -> {tuple(out.shape)}"

tokens = proj(vision_features).detach()                       # (B, 16, LLM_DIM)
with torch.no_grad():
    perm = torch.randperm(N_PATCH)
    assert torch.allclose(resampler(tokens), resampler(tokens[:, perm]), atol=1e-5), \
        "the output depends on the order of the image tokens"
    assert torch.allclose(resampler(tokens)[1:], resampler(tokens[1:]), atol=1e-5), \
        "samples in a batch must not interact"

assert isinstance(resampler.latents, nn.Parameter) and resampler.latents.shape == (NUM_QUERIES, LLM_DIM)
out = resampler(tokens)
(out * torch.randn_like(out)).sum().backward()   # weighted: a LayerNorm output's plain sum has ~0 grad
assert resampler.latents.grad is not None and resampler.latents.grad.abs().sum() > 0, \
    "no gradient reached the latent queries"
n_params = sum(p.numel() for p in resampler.parameters())
print(f"{N_PATCH} image tokens -> {NUM_QUERIES} visual tokens; resampler has {n_params:,} parameters")
print("PerceiverResampler tests passed")

In [ ]:
def merge_image_features(input_ids: torch.Tensor, text_embeds: torch.Tensor,
                         image_features: torch.Tensor, image_token_id: int,
                         attention_mask: torch.Tensor | None = None,
                         ignore_index: int = -100):
    """
    Splice image features into the text embedding sequence at the <image> placeholders.

    Args:
        input_ids:      (B, L) int64; every row contains exactly T copies of image_token_id.
        text_embeds:    (B, L, D) embeddings of input_ids (placeholder rows get overwritten).
        image_features: (B, T, D) projected image tokens; image_features[b, k] goes to the
                        k-th placeholder of sample b.
        image_token_id: id of the <image> placeholder token.
        attention_mask: optional (B, L); 1 = real token, 0 = padding.
        ignore_index:   label value the LM loss skips.

    Returns:
        inputs_embeds: (B, L, D) in text_embeds' dtype -- a new tensor, inputs untouched.
        labels:        (B, L) int64: input_ids with placeholder and padding positions set to
                       ignore_index. NOT shifted.

    Raises:
        ValueError: if any row's number of placeholders differs from T.
    """
    # TODO: (B, L) boolean mask of placeholder positions; validate the per-row count
    # TODO: clone text_embeds, write the (B*T, D) features through the mask (cast the dtype)
    # TODO: labels
    raise NotImplementedError

In [ ]:
T = 4
ids, mask = make_batch(T)
text_embeds = embed_tokens(ids).detach()
# Distinguishable features: every entry of feature k of sample b equals 4*b + k + 1.
feats = torch.arange(1, B * T + 1, dtype=torch.float32).view(B, T, 1).expand(B, T, LLM_DIM)
embeds, labels = merge_image_features(ids, text_embeds, feats, IMAGE_TOKEN_ID, attention_mask=mask)

is_img = ids == IMAGE_TOKEN_ID
assert embeds.shape == text_embeds.shape and embeds.dtype == text_embeds.dtype
for b in range(B):
    pos = is_img[b].nonzero().flatten()
    assert torch.equal(embeds[b, pos, 0], feats[b, :, 0]), \
        f"sample {b}: expected features {feats[b, :, 0].tolist()} at its placeholders, got {embeds[b, pos, 0].tolist()}"
assert torch.equal(embeds[~is_img], text_embeds[~is_img]), "text positions must be untouched"
assert torch.equal(labels, ids.masked_fill(is_img | (mask == 0), -100)), f"wrong labels:\n{labels}"

bad = ids.clone()
bad[0, is_img[0].nonzero()[0]] = 5                 # sample 0 now has T - 1 placeholders
try:
    merge_image_features(bad, text_embeds, feats, IMAGE_TOKEN_ID, attention_mask=mask)
except ValueError as e:
    print("count mismatch raises ValueError:", e)
else:
    raise AssertionError("a placeholder-count mismatch must raise ValueError")
print("merge_image_features tests passed")

In [ ]:
# End to end: ViT features -> connector -> merge -> toy LM head -> next-token loss.
torch.manual_seed(0)
lm_head = nn.Linear(LLM_DIM, VOCAB)

def lm_loss(embeds, labels):
    logits = lm_head(embeds)                                  # (B, L, VOCAB); a real LLM goes here
    # The shift happens HERE: position t predicts token t + 1.
    return F.cross_entropy(logits[:, :-1].reshape(-1, VOCAB), labels[:, 1:].reshape(-1),
                           ignore_index=-100)

for name, connector, n_vis in (("MLP projector", MLPProjector(VISION_DIM, LLM_DIM), N_PATCH),
                               ("projector + resampler",
                                nn.Sequential(MLPProjector(VISION_DIM, LLM_DIM),
                                              PerceiverResampler(LLM_DIM, 8, num_heads=4)), 8)):
    ids, mask = make_batch(n_vis)
    embeds, labels = merge_image_features(ids, embed_tokens(ids), connector(vision_features),
                                          IMAGE_TOKEN_ID, attention_mask=mask)
    loss = lm_loss(embeds, labels)
    loss.backward()
    assert torch.isfinite(loss)
    assert all(p.grad is not None for p in connector.parameters()), "the connector got no gradient"
    n_sup = int((labels[:, 1:] != -100).sum())
    print(f"{name:22s}: sequence length {ids.shape[1]:2d}, {n_vis:2d} visual tokens/image, "
          f"{n_sup} supervised positions, loss {loss.item():.3f}")
print("end-to-end test passed")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("vlm-connector", MLPProjector=MLPProjector, PerceiverResampler=PerceiverResampler,
      merge_image_features=merge_image_features)